In [1]:
import os
import sys

PROJECT_PATH = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_PATH, "srcs")):
    PROJECT_PATH = os.path.dirname(PROJECT_PATH)

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)



In [2]:
from srcs.datasets.utils import setup_hf

HF_CONFIG = setup_hf()

from datasets import load_dataset


from srcs.datasets.vicocktail_dataset import add_validation_split
from srcs.nlp.norm import *
from srcs.nlp.text_transform import TextTransform
from srcs.nlp.tokenizer import WordTokenizer

VALIDATION_FRACTION = 0.03
SEED = 42
MIN_FREQUENCY = 5
VOCAB_DIR = os.path.join(PROJECT_PATH, "srcs", "nlp", "data")
FULL_WORD_VOCAB_PATH = os.path.join(VOCAB_DIR, "full_word_vicocktail_vocab.txt")
FLATTONE_WORD_VOCAB_PATH = os.path.join(VOCAB_DIR, "flattone_word_vicocktail_vocab.txt")

d:\projects\VietnameseVSR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_dataset = load_dataset(
    "nguyenvulebinh/ViCocktail", streaming=False, cache_dir=HF_CONFIG["HF_DATASETS_CACHE"]
)
dataset = add_validation_split(
    raw_dataset, validation_fraction=VALIDATION_FRACTION, seed=SEED
)
train_dataset = dataset["train"]

print("Available splits:", list(dataset))
print("Train samples:", len(train_dataset))
print("Validation samples:", len(dataset["validation"]))
print("Test samples:", len(dataset["test"]))

Available splits: ['train', 'validation', 'test', 'test_snr_n5_interferer_1', 'test_snr_n5_interferer_2', 'test_snr_0_interferer_1', 'test_snr_0_interferer_2', 'test_snr_5_interferer_1', 'test_snr_5_interferer_2', 'test_snr_10_interferer_1', 'test_snr_10_interferer_2']
Train samples: 188229
Validation samples: 5844
Test samples: 1167


In [4]:
def decode_label(label: object) -> str:
    if isinstance(label, (bytes, bytearray, memoryview)):
        return bytes(label).decode("utf-8")
    return str(label)


def iter_transcripts(dataset_split):
    for label in dataset_split["label"]:
        transcript = decode_label(label).strip()
        if transcript:
            yield transcript


In [5]:
common_rules = [
    RemovePunctNormalizer(),
    RemoveNumericNormalizer(),
    RemoveSpecialCharNormalizer(),
    SpaceNormalizer(),
]
full_word_normalizer = TextNormalizer(
    lowercase=True,
    rules=common_rules,
)
flattone_word_normalizer = TextNormalizer(
    lowercase=True,
    rules=[
        RemovePunctNormalizer(),
        RemoveNumericNormalizer(),
        RemoveSpecialCharNormalizer(),
        FlatToneNormalizer(),
        SpaceNormalizer(),
    ],
)

full_word_transform = TextTransform(
    tokenizer=WordTokenizer(full_word_normalizer),
    vocab_path=FULL_WORD_VOCAB_PATH,
)
flattone_word_transform = TextTransform(
    tokenizer=WordTokenizer(flattone_word_normalizer),
    vocab_path=FLATTONE_WORD_VOCAB_PATH,
)


In [6]:
full_word_path = full_word_transform.create_vocab(
    iter_transcripts(train_dataset),
    min_frequency=MIN_FREQUENCY,
)
flattone_word_path = flattone_word_transform.create_vocab(
    iter_transcripts(train_dataset),
    min_frequency=MIN_FREQUENCY,
)

print("Full-word vocabulary path:", full_word_path)
print("Full-word vocabulary size:", len(full_word_transform.token_list))
print("Flattone-word vocabulary path:", flattone_word_path)
print("Flattone-word vocabulary size:", len(flattone_word_transform.token_list))


Full-word vocabulary path: d:\projects\VietnameseVSR\srcs\nlp\data\full_word_vicocktail_vocab.txt
Full-word vocabulary size: 3897
Flattone-word vocabulary path: d:\projects\VietnameseVSR\srcs\nlp\data\flattone_word_vicocktail_vocab.txt
Flattone-word vocabulary size: 2007


In [8]:
for name, transform in (
    ("full-word", full_word_transform),
    ("flattone-word", flattone_word_transform),
):
    assert transform.token_list[0] == transform.blank_token
    assert transform.token_list[1] == transform.unknown_token
    assert len(transform.token_list) == len(set(transform.token_list))

    print(f"{name} first tokens:", transform.token_list[:20])

print("Vocabulary validation passed.")


full-word first tokens: ['<blank>', '<unk>', 'là', 'cái', 'mình', 'có', 'thì', 'mà', 'một', 'không', 'bạn', 'và', 'những', 'nó', 'của', 'người', 'ta', 'các', 'sẽ', 'đó']
flattone-word first tokens: ['<blank>', '<unk>', 'la', 'cai', 'minh', 'co', 'thi', 'ma', 'nhưng', 'ban', 'môt', 'không', 'va', 'no', 'cua', 'ngươi', 'se', 'thê', 'ta', 'cac']
Vocabulary validation passed.
